# Tech Challenge Fase 4 — Treino LSTM no Google Colab

Este notebook executa o fluxo completo:

1. Clonar o repositório GitHub
2. Instalar dependências
3. Treinar o modelo LSTM
4. Gerar artefatos em `models/`
5. Testar a API
6. Fazer commit e push para o GitHub

## 1. Configurações

Edite as variáveis abaixo antes de executar.

In [ ]:
GITHUB_USERNAME = "SEU_USUARIO"
GITHUB_REPO = "SEU_REPOSITORIO"
BRANCH = "main"

SYMBOL = "DIS"
START_DATE = "2018-01-01"
END_DATE = "2024-07-20"
SEQUENCE_LENGTH = 60
EPOCHS = 40
BATCH_SIZE = 32

## 2. Clonar repositório

Se o repositório ainda não existe no Colab, execute esta célula.

In [ ]:
import os
repo_url = f"https://github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git"

if not os.path.exists(GITHUB_REPO):
    !git clone -b {BRANCH} {repo_url}
else:
    print("Repositório já existe no Colab.")

%cd /content/{GITHUB_REPO}

## 3. Instalar dependências

In [ ]:
!python -m pip install --upgrade pip
!pip install -r requirements.txt

## 4. Treinar modelo LSTM

In [ ]:
!python -m src.training.train \
  --symbol {SYMBOL} \
  --start-date {START_DATE} \
  --end-date {END_DATE} \
  --sequence-length {SEQUENCE_LENGTH} \
  --epochs {EPOCHS} \
  --batch-size {BATCH_SIZE} \
  --model-dir models \
  --report-dir reports

## 5. Visualizar métricas

In [ ]:
import json

with open("reports/metrics.json", "r", encoding="utf-8") as f:
    metrics = json.load(f)

metrics

## 6. Conferir artefatos

In [ ]:
!ls -lh models
!ls -lh reports

## 7. Testar API rapidamente

Esta célula inicia a API em background no Colab.

In [ ]:
import subprocess
import time

server = subprocess.Popen(
    ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

time.sleep(8)
print("API iniciada em background.")

Teste do endpoint `/health`.

In [ ]:
!curl -s http://localhost:8000/health | python -m json.tool

Teste do endpoint `/predict/from-yfinance`.

In [ ]:
!curl -s -X POST "http://localhost:8000/predict/from-yfinance" \
  -H "Content-Type: application/json" \
  -d @examples/predict_from_yfinance_request.json | python -m json.tool

Teste do endpoint `/metrics`.

In [ ]:
!curl -s http://localhost:8000/metrics | python -m json.tool

## 8. Commit e push para GitHub

Use um token do GitHub. **Nunca salve o token em arquivo do projeto.**

No Colab, você pode preencher o token temporariamente em variável de ambiente.

In [ ]:
import os
from getpass import getpass

os.environ["GIT_NAME"] = "Seu Nome"
os.environ["GIT_EMAIL"] = "seu-email@gmail.com"
os.environ["GITHUB_USERNAME"] = GITHUB_USERNAME
os.environ["GITHUB_REPO"] = GITHUB_REPO
os.environ["GIT_BRANCH"] = BRANCH
os.environ["COMMIT_MESSAGE"] = "Add trained LSTM model artifacts"

if "GITHUB_TOKEN" not in os.environ:
    os.environ["GITHUB_TOKEN"] = getpass("Cole seu GitHub token: ")

In [ ]:
!bash scripts/git_push_from_colab.sh

## 9. Próximo passo

Depois do push, vá ao Render, conecte o repositório e faça o deploy usando Docker ou Blueprint.